# XGBoost (eXtreme Gradient Boosting) - Wine Classification

## What is XGBoost?

**XGBoost** stands for **eXtreme Gradient Boosting**. It is an advanced implementation of gradient boosting algorithm that is highly efficient, flexible, and portable.

### Key Concepts:

1. **Gradient Boosting**: A machine learning technique that builds a model in a stage-wise manner by combining multiple weak learners (typically decision trees) to create a strong predictive model.

2. **Boosting**: An ensemble technique where new models are trained to correct the errors made by previous models. Unlike bagging (Random Forest), boosting builds models sequentially.

3. **Why XGBoost?**
   - **Speed**: Faster than traditional gradient boosting due to parallel processing
   - **Performance**: Often produces better results with less overfitting
   - **Regularization**: Built-in L1 and L2 regularization to prevent overfitting
   - **Handling Missing Values**: Can automatically handle missing data
   - **Tree Pruning**: Uses depth-first approach with max_depth parameter

### Common Use Cases:
- Classification problems (spam detection, disease prediction)
- Regression problems (price prediction, sales forecasting)
- Ranking problems (search engines, recommendation systems)
- Winning solution in many Kaggle competitions

---

## Problem Statement

In this example, we will use XGBoost to classify wines into different types based on their chemical properties. We'll use the wine dataset from sklearn, which contains measurements of 13 different chemical properties of wines from three different cultivars (wine types/origins) in Italy.

## Step 1: Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Machine Learning libraries
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## Step 2: Load and Explore the Dataset

In [ ]:
# Load the wine dataset
wine = load_wine()

# Create a DataFrame
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Display dataset information
print("Dataset Info:")
print(df.info())

print("\nTarget Classes:")
print(df['target'].value_counts())

print("\nDataset Description:")
print(wine.DESCR)

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

## Step 3: Data Visualization

In [ ]:
# Plot class distribution
plt.figure(figsize=(8, 5))
df['target'].value_counts().plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title('Distribution of Wine Classes', fontsize=14, fontweight='bold')
plt.xlabel('Wine Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (select first 5 features for clarity)
plt.figure(figsize=(10, 8))
correlation_matrix = df.iloc[:, :7].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Wine Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Prepare Data for Training

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Split the data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

In [ ]:
# Feature scaling (optional for XGBoost but can help)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

## Step 5: Train XGBoost Model

### Key XGBoost Parameters:

- **n_estimators**: Number of boosting rounds (trees to build)
- **max_depth**: Maximum depth of each tree (prevents overfitting)
- **learning_rate**: Step size shrinkage to prevent overfitting (eta)
- **objective**: Learning task (e.g., 'multi:softmax' for multiclass classification)
- **eval_metric**: Metric to evaluate model performance
- **random_state**: Seed for reproducibility

In [ ]:
# Create XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=100,        # Number of trees
    max_depth=5,             # Maximum tree depth
    learning_rate=0.1,       # Learning rate (eta)
    objective='multi:softmax',  # Multiclass classification
    num_class=3,             # Number of classes
    random_state=42,
    eval_metric='mlogloss'   # Multiclass log loss
)

print("XGBoost Model Created!")
print("\nModel Parameters:")
print(xgb_model.get_params())

In [ ]:
# Train the model
xgb_model.fit(X_train_scaled, y_train)
print("Model Training Completed!")

## Step 6: Model Evaluation

In [ ]:
# Make predictions
y_pred_train = xgb_model.predict(X_train_scaled)
y_pred_test = xgb_model.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Testing Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Detailed classification report
print("Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_test, target_names=wine.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=wine.target_names, 
            yticklabels=wine.target_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - XGBoost Wine Classification', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix:")
print(cm)

## Step 7: Feature Importance Analysis

One of the advantages of XGBoost is that it provides feature importance scores, helping us understand which features contribute most to the predictions.

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Feature Importance - XGBoost Model', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost's built-in feature importance plot
plt.figure(figsize=(10, 6))
xgb.plot_importance(xgb_model, max_num_features=10, height=0.5, importance_type='weight')
plt.title('Feature Importance (using XGBoost built-in plot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8: Making Predictions on New Data

In [ ]:
# Select a few test samples
sample_indices = [0, 10, 20]
samples = X_test.iloc[sample_indices]
samples_scaled = scaler.transform(samples)

# Make predictions
predictions = xgb_model.predict(samples_scaled)

# Display results
print("Sample Predictions:")
print("="*60)
for i, idx in enumerate(sample_indices):
    actual = y_test.iloc[idx]
    predicted = predictions[i]
    print(f"Sample {i+1}:")
    print(f"  Actual Class: {wine.target_names[actual]}")
    print(f"  Predicted Class: {wine.target_names[predicted]}")
    print(f"  Match: {'✓ Yes' if actual == predicted else '✗ No'}")
    print("-"*60)

## Summary

### What We Learned:

1. **XGBoost Basics**: Understanding what XGBoost is and why it's powerful
2. **Data Preparation**: Loading, exploring, and preprocessing data
3. **Model Training**: Creating and training an XGBoost classifier with key parameters
4. **Model Evaluation**: Using accuracy, classification report, and confusion matrix
5. **Feature Importance**: Identifying which features are most important for predictions
6. **Making Predictions**: Using the trained model to classify new wine samples

### Key Advantages of XGBoost:

- ✅ High Performance: Often achieves better results than other algorithms
- ✅ Speed: Faster training due to parallel processing
- ✅ Regularization: Built-in L1 and L2 regularization prevents overfitting
- ✅ Feature Importance: Automatically identifies important features
- ✅ Flexibility: Works well with various types of data
- ✅ Missing Values: Can handle missing data automatically

### When to Use XGBoost:

- Structured/tabular data (not images or text directly)
- Classification and regression problems
- When model interpretability is important (feature importance)
- Competitive machine learning (Kaggle competitions)
- When you need high accuracy with reasonable training time

---

**Note**: XGBoost is widely used in industry and competitions due to its excellent performance and flexibility. It's an essential tool in any data scientist's toolkit!